# Logistic Regression Tuning (Random Dev + OOT)
## Loan-default Probability-of-Default (PD) scorecard

Same **Train–Val–Test–OOT** pipeline as `02b_model_tuning_random_dev_oot.ipynb`, focused on tuning and selecting the best **L2-regularised Logistic Regression** model only.

LR is the regulator-friendly **scorecard** baseline: coefficients map directly to points for adverse-action notices and audit. We tune inverse regularisation strength `C` on the validation set (primary metric: **ROC-AUC / Gini**) and export to `model_store/best_params_lr_random_dev.json`.


In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, f1_score
from sklearn.model_selection import PredefinedSplit
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.base import clone

from utils.config import COST_FALSE_NEGATIVE, COST_FALSE_POSITIVE, MODEL_STORE_DIR
from utils.ml.dataset import (
    load_training_frame, select_feature_columns, coerce_feature_matrix,
    random_dev_oot_split, ks_statistic,
)
from utils.ml.train import _build_candidates, _evaluate, _best_cost_threshold

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
LR_PARAMS_OUT = MODEL_STORE_DIR / "best_params_lr_random_dev.json"
print("Repo root:", ROOT)


## 1. Load the point-in-time training table

`g2_training_table` is the **leakage-safe** modelling table from the medallion pipeline. Each row is one loan joined to the **exact `g2_feature_store` row that inference scores** — the feature snapshot **as-of the application month** — so `feature_snapshot_date = loan_start_date < label_snapshot_date`. The label is `30dpd@mob6`, observed 6 months after disbursement.

Imputation and **StandardScaler** (required for LR on mixed numeric + one-hot features) are fit *inside* the sklearn pipeline on the training fold only — no train/test contamination.


In [ ]:
AS_OF = "2024-12-01"
df = load_training_frame(AS_OF)

feature_cols = select_feature_columns(df)
df["loan_start_date"] = pd.to_datetime(df["loan_start_date"])
df["label_snapshot_date"] = pd.to_datetime(df["label_snapshot_date"])
df["feature_snapshot_date"] = pd.to_datetime(df["feature_snapshot_date"])

assert (df["feature_snapshot_date"] <= df["loan_start_date"]).all(), "feature leakage detected!"

print(f"Rows (labelled loans)  : {len(df):,}")
print(f"Model features         : {len(feature_cols)}")
print(f"Application cohorts     : {df['loan_start_date'].nunique()} "
      f"({df['loan_start_date'].min():%Y-%m} -> {df['loan_start_date'].max():%Y-%m})")
print(f"Overall bad rate        : {df['default_flag'].mean():.3f}")
print("Feature anchor == application month for all rows:",
      bool((df["feature_snapshot_date"] == df["loan_start_date"]).all()))
df[["loan_id", "loan_start_date", "feature_snapshot_date", "label_snapshot_date", "default_flag"]].head()


## 2. Why Logistic Regression for this use case

| Property | Value | Implication for LR |
|---|---|---|
| Labelled rows | ~9k | Sufficient for a regularised linear scorecard; too small for deep learning. |
| Features | ~112 | Wide matrix (one-hot categoricals + numerics). **L2 penalty + scaling** essential. |
| Bad rate | ~29% | Moderate imbalance — `class_weight='balanced'` stabilises coefficients. |
| Cohort bad rate | stable 26–32% | Stable prior; a single LR champion generalises across months. |

**Framing.** LR produces an interpretable **scorecard** (coefficients → points) — useful for adverse action and regulatory review — while tree challengers (LightGBM in `02b`) maximise discrimination. Champion selection in the deck is by **validation ROC-AUC** with a side-by-side comparison table.

**Primary metric: ROC-AUC / Gini (`Gini = 2·AUC − 1`).** We also report KS, PR-AUC, and Brier score. LR will not beat LightGBM on AUC alone; the value is explainability and industry-standard scorecard format.


In [ ]:
cohort = (df.groupby(df["loan_start_date"].dt.to_period("M"))
            .agg(n=("default_flag", "size"), bad_rate=("default_flag", "mean")))
cohort.index = cohort.index.astype(str)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
cohort["n"].plot(kind="bar", ax=ax[0], color="#4C72B0")
ax[0].set_title("Labelled loans per application cohort"); ax[0].set_ylabel("# loans"); ax[0].set_xlabel("")
cohort["bad_rate"].plot(marker="o", ax=ax[1], color="#C44E52")
ax[1].axhline(df["default_flag"].mean(), ls="--", color="grey", label="overall")
ax[1].set_title("Bad rate (30dpd@mob6) by application cohort"); ax[1].set_ylabel("bad rate"); ax[1].set_ylim(0, 0.45)
ax[1].legend(); plt.tight_layout(); plt.show()
cohort.round(3)


## 3. Train–Val–Test–OOT split

1. **OOT (temporal)** — last **3** application cohorts, sealed until the final headline score.
2. **Train / Val / Test (random)** — stratified 70 / 15 / 15 on all older cohorts.

At ~7.5k development rows with dedicated val (~1.1k), test (~1.1k), and OOT (~1.5k), **k-fold CV is not required**. Section 4 evaluates each `C` on the single train→val split.


In [ ]:
OOT_COHORTS = 3
TEST_FRAC = 0.15
VAL_FRAC = 0.15

train_df, val_df, test_df, oot_df = random_dev_oot_split(
    df,
    n_oot_cohorts=OOT_COHORTS,
    test_size=TEST_FRAC,
    val_size=VAL_FRAC,
    random_state=RANDOM_STATE,
)

def xy(frame):
    return coerce_feature_matrix(frame, feature_cols), frame["default_flag"].astype(int)

X_train, y_train = xy(train_df)
X_val,   y_val   = xy(val_df)
X_test,  y_test  = xy(test_df)
X_oot,   y_oot   = xy(oot_df)

X_dev = pd.concat([X_train, X_val]); y_dev = pd.concat([y_train, y_val])

for name, frame in [("train", train_df), ("val", val_df), ("test", test_df), ("oot", oot_df)]:
    cohorts = sorted(frame["loan_start_date"].dt.strftime("%Y-%m").unique())
    print(
        f"{name:>5}: n={len(frame):5d}  bad_rate={frame['default_flag'].mean():.3f}  "
        f"cohorts={cohorts[0]}..{cohorts[-1]} ({len(cohorts)} months)"
    )


## 4. Baseline Logistic Regression (default `C = 1.0`)

Pipeline: median-impute → **StandardScaler** → `LogisticRegression(penalty='l2', class_weight='balanced')`. Fitted on `train`, scored on `validation`.


In [ ]:
baseline_pipe = _build_candidates(feature_cols, tuned={})["logistic_regression"]
baseline_pipe.fit(X_train, y_train)

tr = _evaluate("logistic_regression", y_train, baseline_pipe.predict_proba(X_train)[:, 1])
va = _evaluate("logistic_regression", y_val, baseline_pipe.predict_proba(X_val)[:, 1])

baseline_row = {
    "C": 1.0,
    "train_auc": tr["roc_auc"], "val_auc": va["roc_auc"],
    "auc_gap": tr["roc_auc"] - va["roc_auc"],
    "val_gini": va["gini"], "val_ks": va["ks"],
    "val_pr_auc": va["pr_auc"], "val_brier": va["brier"],
}
baseline_metrics = pd.DataFrame([baseline_row]).set_index("C")
print("Baseline (C=1.0):")
baseline_metrics.round(4)


## 5. L2 regularisation tuning — select best `C`

With ~30% bad rate and ~112 features (including one-hot expansions), LR needs **L2 regularisation**. We grid-search **`C`** on a log scale (smaller `C` = stronger penalty):

| `C` | Interpretation |
|---|---|
| 0.001 | Very strong shrinkage — underfit risk |
| 0.01 | Strong |
| 0.1 | Moderate |
| 1.0 | sklearn default (Section 4 baseline) |
| 10 | Weak penalty — overfit risk on wide matrix |

**Selection rule:** highest validation ROC-AUC; tie-break on smaller train–val AUC gap (LR rarely overfits badly at this scale).


In [ ]:
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]

rows = []
fitted_by_c = {}
for c in C_GRID:
    pipe = _build_candidates(feature_cols, tuned={"logistic_regression": {"C": c}})["logistic_regression"]
    pipe.fit(X_train, y_train)
    fitted_by_c[c] = pipe
    tr = _evaluate("logistic_regression", y_train, pipe.predict_proba(X_train)[:, 1])
    va = _evaluate("logistic_regression", y_val, pipe.predict_proba(X_val)[:, 1])
    rows.append({
        "C": c,
        "train_auc": tr["roc_auc"], "val_auc": va["roc_auc"],
        "auc_gap": tr["roc_auc"] - va["roc_auc"],
        "val_gini": va["gini"], "val_ks": va["ks"],
        "val_pr_auc": va["pr_auc"], "val_brier": va["brier"],
    })

tuning_metrics = (
    pd.DataFrame(rows).set_index("C")
    .sort_values(["val_auc", "auc_gap"], ascending=[False, True])
)

best_c = float(tuning_metrics.index[0])
best_params = {"C": best_c, "penalty": "l2"}
best_pipe = fitted_by_c[best_c]
best_pipe.fit(X_dev, y_dev)  # refit on train+val for downstream sections

print(f"Best C = {best_c}  (val AUC = {tuning_metrics.loc[best_c, 'val_auc']:.4f})")
print(f"Baseline C=1.0 val AUC = {baseline_row['val_auc']:.4f}")
print(f"Lift vs baseline: {tuning_metrics.loc[best_c, 'val_auc'] - baseline_row['val_auc']:+.4f}")
tuning_metrics.round(4)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
tuning_metrics[["train_auc", "val_auc"]].plot(marker="o", ax=ax[0])
ax[0].set_xscale("log"); ax[0].set_xlabel("C (inverse regularisation strength)")
ax[0].set_ylabel("ROC-AUC"); ax[0].set_title("Train vs validation AUC by C")
ax[0].legend(["train", "validation"])

tuning_metrics["val_gini"].plot(marker="s", color="#C44E52", ax=ax[1])
ax[1].set_xscale("log"); ax[1].set_xlabel("C")
ax[1].set_ylabel("Gini"); ax[1].set_title("Validation Gini by C")
plt.tight_layout(); plt.show()


In [ ]:
# Random test-fold check (in-sample generalisation, not used for C selection)
test_score = best_pipe.predict_proba(X_test)[:, 1]
holdout = _evaluate("logistic_regression_tuned", y_test, test_score)
print("Random test-fold (tuned LR, refit on train+val):")
for k in ["roc_auc", "gini", "ks", "pr_auc", "brier"]:
    print(f"  {k:>8}: {holdout[k]:.4f}")

# Sealed OOT — headline temporal performance
oot_score = best_pipe.predict_proba(X_oot)[:, 1]
oot = _evaluate("logistic_regression_tuned", y_oot, oot_score)
print("\nOut-of-time test (sealed cohorts):")
for k in ["roc_auc", "gini", "ks", "pr_auc", "brier"]:
    print(f"  {k:>8}: {oot[k]:.4f}")


In [ ]:
def _to_jsonable(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    return v

export = {"logistic_regression": {k: _to_jsonable(v) for k, v in best_params.items()}}
MODEL_STORE_DIR.mkdir(parents=True, exist_ok=True)
LR_PARAMS_OUT.write_text(json.dumps(export, indent=2), encoding="utf-8")
print(f"Wrote tuned LR params -> {LR_PARAMS_OUT}")
print(json.dumps(export, indent=2))


## 6. Probability calibration

Wrap the tuned LR in `CalibratedClassifierCV` (isotonic) so scores read as true PDs for expected-loss and provisioning. Compare reliability on the random test fold.


In [ ]:
calibrated = CalibratedClassifierCV(best_pipe, method="isotonic", cv=3)
calibrated.fit(X_dev, y_dev)

raw_p = best_pipe.predict_proba(X_test)[:, 1]
cal_p = calibrated.predict_proba(X_test)[:, 1]

plt.figure(figsize=(6, 6))
for label, p in [("uncalibrated", raw_p), ("isotonic", cal_p)]:
    frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    plt.plot(mean_pred, frac_pos, marker="o", label=f"{label} (Brier={brier_score_loss(y_test, p):.3f})")
plt.plot([0, 1], [0, 1], "--", color="grey", label="perfectly calibrated")
plt.xlabel("Mean predicted PD"); plt.ylabel("Observed default rate")
plt.title("Reliability curve (random test fold)"); plt.legend(); plt.show()


## 7. Decision threshold from business cost

Choose the PD cut-off minimising expected cost on validation (FN:FP = 5:1). A ranking model is not enough for approve/decline — the threshold encodes business policy.


In [ ]:
val_p = calibrated.predict_proba(X_val)[:, 1]
threshold, exp_cost = _best_cost_threshold(y_val, val_p)
print(f"Cost-optimal threshold={threshold:.3f} (FN:FP cost = {COST_FALSE_NEGATIVE}:{COST_FALSE_POSITIVE})")

grid = np.linspace(0.05, 0.95, 181)
appr, bad_appr, costs = [], [], []
for thr in grid:
    y_pred = (val_p >= thr).astype(int)
    appr.append((val_p < thr).mean())
    bad_appr.append(y_val[val_p < thr].mean() if (val_p < thr).any() else np.nan)
    fn = int(np.sum((y_pred == 0) & (y_val == 1)))
    fp = int(np.sum((y_pred == 1) & (y_val == 0)))
    costs.append(COST_FALSE_NEGATIVE * fn + COST_FALSE_POSITIVE * fp)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(grid, costs, color="#8172B3")
ax[0].axvline(threshold, ls="--", color="red")
ax[0].set_title("Expected business cost vs threshold (validation)"); ax[0].set_xlabel("PD threshold")
ax[1].plot(appr, bad_appr, marker=".", color="#C44E52")
ax[1].axhline(y_test.mean(), ls="--", color="grey", label=f"portfolio bad rate ({y_test.mean():.2f})")
ax[1].set_title("Approval rate vs bad-rate among approved (validation)")
ax[1].set_xlabel("approval rate"); ax[1].set_ylabel("bad rate among approved"); ax[1].legend()
plt.tight_layout(); plt.show()

approve = val_p < threshold
print(f"At chosen threshold -> approval rate={approve.mean():.1%}, "
      f"bad rate among approved={y_val[approve].mean():.1%} (vs {y_val.mean():.1%} portfolio)")


## 8. Scorecard interpretability — coefficients & permutation importance

**Coefficients** (after scaling) show direction and relative weight — the raw material for a points-based scorecard. **Permutation importance** on validation ranks features by AUC drop when shuffled (model-agnostic sanity check).


In [ ]:
model = calibrated.calibrated_classifiers_[0].estimator.named_steps["model"]
pre = calibrated.calibrated_classifiers_[0].estimator.named_steps["preprocessor"]
feat_names = pre.get_feature_names_out()

coefs = pd.Series(model.coef_.ravel(), index=feat_names).sort_values(key=abs, ascending=False)
top_coef = coefs.head(20)

plt.figure(figsize=(8, 6))
colors = ["#C44E52" if v > 0 else "#4C72B0" for v in top_coef.values]
top_coef.iloc[::-1].plot(kind="barh", color=colors[::-1])
plt.title("Top 20 LR coefficients (scaled features, + = higher default risk)")
plt.xlabel("coefficient"); plt.tight_layout(); plt.show()
top_coef.round(4)


In [ ]:
perm = permutation_importance(
    best_pipe, X_val, y_val, scoring="roc_auc",
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
)
imp = (pd.Series(perm.importances_mean, index=feature_cols)
         .sort_values(ascending=False).head(20))

plt.figure(figsize=(8, 6))
imp.iloc[::-1].plot(kind="barh", color="#55A868")
plt.title("Top 20 features by permutation importance (AUC drop, validation)")
plt.xlabel("mean AUC decrease"); plt.tight_layout(); plt.show()
imp.round(4)


## 9. Fit diagnostics — learning curve

Train vs cross-validated AUC as training size grows. LR typically shows a small train–val gap when `C` is well chosen.


In [ ]:
from sklearn.model_selection import learning_curve, StratifiedKFold

sizes, train_sc, val_sc = learning_curve(
    clone(best_pipe), X_dev, y_dev, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    train_sizes=np.linspace(0.3, 1.0, 6), scoring="roc_auc", n_jobs=-1, shuffle=True,
)
tr_m, tr_s = train_sc.mean(1), train_sc.std(1)
va_m, va_s = val_sc.mean(1), val_sc.std(1)

plt.figure(figsize=(7, 4.5))
plt.plot(sizes, tr_m, "o-", color="#4C72B0", label="train AUC")
plt.fill_between(sizes, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color="#4C72B0")
plt.plot(sizes, va_m, "o-", color="#C44E52", label="CV AUC")
plt.fill_between(sizes, va_m - va_s, va_m + va_s, alpha=0.15, color="#C44E52")
plt.xlabel("training samples"); plt.ylabel("ROC-AUC")
plt.title(f"Learning curve — tuned LR (C={best_c})")
gap = tr_m[-1] - va_m[-1]
print(f"Final train-CV AUC gap = {gap:.3f} -> {'well-fit' if gap < 0.03 else 'check regularisation'}")
plt.legend(); plt.tight_layout(); plt.show()


## 10. Model card & handoff

| Field | Value |
|---|---|
| **Model family** | L2 Logistic Regression (scorecard) |
| **Selection metric** | Validation ROC-AUC / Gini |
| **Best `C`** | Printed above |
| **Calibration** | Isotonic (`CalibratedClassifierCV`) |
| **Decision rule** | Decline if calibrated PD ≥ cost-optimal threshold |
| **Dev split** | Random train/val/test + temporal OOT (3 cohorts) |

### Promote to production
1. `model_store/best_params_lr_random_dev.json` was written in Section 5.
2. For a full bake-off champion, compare this notebook's OOT AUC to `02b` LightGBM results in the deck.
3. To train the LR champion in the pipeline, copy params to `best_params.json` or extend `utils/ml/train.py` to read the LR-specific file, then run `python -m utils.ml.train --snapshotdate 2024-12-01 --force-retrain`.


In [ ]:
summary = {
    "model_family": "logistic_regression",
    "best_C": best_c,
    "val_roc_auc": round(float(tuning_metrics.loc[best_c, "val_auc"]), 4),
    "oot_roc_auc": round(float(oot["roc_auc"]), 4),
    "oot_gini": round(float(oot["gini"]), 4),
    "decision_threshold": round(float(threshold), 3),
    "params_file": str(LR_PARAMS_OUT),
    "best_params": best_params,
}
print(json.dumps(summary, indent=2))
